In [1]:
import itertools
import pickle
import re
import traceback
from datetime import datetime
from pathlib import Path
from typing import Iterable, Literal

import numpy as np
import pandas as pd

from HPC_2P_analysis.utils.config import (
    select_files,
    get_itr_index,
    index_to_type,
    get_data_track_type,
    get_data_behavior_type,
)

from HPC_2P_analysis.utils.loadData import load_raw_data

from HPC_2P_analysis.utils.processData import (
    align_track_hpc,
    apply_neuron_mask_to_attracted_data,
)

In [2]:
def extract_session_date_from_name(name: str | Path) -> str | None:
    """
    Extract session date from file name.

    Supported examples
    ------------------
    neuro_type_saveHP01_1_2024-12-13_first-pattern.mat
        -> 2024-12-13

    neuro_type_saveHP01_1_20241213_pattern.mat
        -> 2024-12-13

    neuro_type_saveHP01_1_241213_pattern.mat
        -> 2024-12-13
    """
    name = Path(name).name

    match = re.search(
        r"(?<!\d)((?:19|20)\d{2})[-_]?([01]\d)[-_]?([0-3]\d)(?!\d)",
        name,
    )

    if match is not None:
        year, month, day = match.groups()
        date_str = f"{year}-{month}-{day}"
        datetime.strptime(date_str, "%Y-%m-%d")
        return date_str

    match = re.search(
        r"(?<!\d)(\d{2})([01]\d)([0-3]\d)(?!\d)",
        name,
    )

    if match is not None:
        yy, month, day = match.groups()
        year = f"20{yy}"
        date_str = f"{year}-{month}-{day}"
        datetime.strptime(date_str, "%Y-%m-%d")
        return date_str

    return None


def session_date_to_key(session_date: str) -> str:
    """
    Convert session date to compact key.

    Example
    -------
    2024-12-13 -> 20241213
    """
    return str(session_date).replace("-", "")


def add_session_info_to_file_info(
    file_info: dict,
    file_path: str | Path | None = None,
    *,
    require_session_date: bool = True,
) -> dict:
    """
    Add session_date and session_key to file_info.

    This modifies metadata only. It does not touch firing, index, or neuron data.
    """
    file_info = dict(file_info)
    session_date = file_info.get("session_date", None)

    if session_date is None:
        candidates = []

        if "file_name" in file_info:
            candidates.append(file_info["file_name"])

        if "source_stem" in file_info:
            candidates.append(file_info["source_stem"])

        if "file_path" in file_info:
            candidates.append(file_info["file_path"])

        if file_path is not None:
            candidates.append(file_path)

        for candidate in candidates:
            session_date = extract_session_date_from_name(candidate)

            if session_date is not None:
                break

    if session_date is None:
        if require_session_date:
            raise ValueError(
                "Cannot extract session date from file name. "
                "Expected one of: 2024-12-13, 20241213, or 241213.\n"
                f"file_info={file_info}\n"
                f"file_path={file_path}"
            )

        file_info["session_date"] = None
        file_info["session_key"] = None
        return file_info

    file_info["session_date"] = session_date
    file_info["session_key"] = session_date_to_key(session_date)

    return file_info


def add_session_columns_to_file_df(
    file_df: pd.DataFrame,
    *,
    require_session_date: bool = True,
) -> pd.DataFrame:
    """
    Add session_date and session_key columns to file_df.
    """
    df = file_df.copy()

    if "file_name" not in df.columns:
        df["file_name"] = df["file_path"].map(lambda x: Path(x).name)

    session_dates = []

    for _, row in df.iterrows():
        session_date = None

        for col in ["file_name", "source_stem", "file_path"]:
            if col in row and pd.notna(row[col]):
                session_date = extract_session_date_from_name(row[col])

                if session_date is not None:
                    break

        session_dates.append(session_date)

    df["session_date"] = session_dates
    df["session_key"] = [
        session_date_to_key(x) if x is not None else None
        for x in session_dates
    ]

    if require_session_date and df["session_date"].isna().any():
        bad = df.loc[
            df["session_date"].isna(),
            ["file_name", "file_path"],
        ]

        raise ValueError(
            "Some files do not contain a parsable session date.\n"
            f"{bad.to_string(index=False)}"
        )

    return df

In [3]:
Alternative = Literal["greater", "two-sided", "less"]
BaselineMethod = Literal["median", "mean", "none"]
RuleCorrection = Literal["none", "bonferroni"]


def make_screen_save_path(
    output_root: str | Path,
    file_info: dict,
) -> Path:
    """
    Make output path for one screening result.

    Full reward:
        output_root/mouse_id/mouse_id_session-date_task-type_screen.pkl

    75% reward:
        output_root/mouse_id/mouse_id_session-date_75_task-type_screen.pkl
    """
    output_root = Path(output_root)

    file_info = add_session_info_to_file_info(
        file_info,
        file_info.get("file_path", None),
        require_session_date=True,
    )

    mouse_id = file_info["mouse_id"]
    task_type = file_info["task_type"]
    session_date = file_info["session_date"]
    reward_mode = file_info.get("reward_mode", "full")

    if task_type is None:
        raise ValueError("file_info['task_type'] is None.")

    if session_date is None:
        raise ValueError("file_info['session_date'] is None.")

    if reward_mode not in {"full", "75"}:
        raise ValueError(
            f"Unknown reward_mode={reward_mode!r}. "
            "Expected 'full' or '75'."
        )

    save_dir = output_root / mouse_id

    if reward_mode == "75":
        save_name = f"{mouse_id}_{session_date}_75_{task_type}_screen.pkl"
    else:
        save_name = f"{mouse_id}_{session_date}_{task_type}_screen.pkl"

    return save_dir / save_name

In [4]:
def merge_position_bins(
    fr: np.ndarray,
    n_merged_bins: int = 40,
) -> np.ndarray:
    """
    Merge spatial bins by averaging adjacent bins.

    Input shape
    -----------
    (n_neuron, n_trial, n_position)

    Output shape
    ------------
    (n_neuron, n_trial, n_merged_bins)
    """
    if fr.ndim != 3:
        raise ValueError(
            f"Expected fr with shape (n_neuron, n_trial, n_position), "
            f"got {fr.shape}."
        )

    n_pos = fr.shape[2]

    if not (1 <= n_merged_bins <= n_pos):
        raise ValueError(
            f"n_merged_bins should be in [1, {n_pos}], "
            f"got {n_merged_bins}."
        )

    bin_groups = np.array_split(np.arange(n_pos), n_merged_bins)

    merged = np.empty(
        (fr.shape[0], fr.shape[1], len(bin_groups)),
        dtype=float,
    )

    for bin_idx, pos_idx in enumerate(bin_groups):
        merged[:, :, bin_idx] = np.nanmean(fr[:, :, pos_idx], axis=2)

    return merged


def trial_baseline_correct(
    fr: np.ndarray,
    method: BaselineMethod = "median",
) -> np.ndarray:
    """
    Subtract each trial's spatial baseline.

    method
    ------
    "median":
        subtract median across position bins for each neuron-trial.

    "mean":
        subtract mean across position bins for each neuron-trial.

    "none":
        no baseline correction.
    """
    if method == "none":
        return fr.astype(float, copy=True)

    if method == "median":
        baseline = np.nanmedian(fr, axis=2, keepdims=True)

    elif method == "mean":
        baseline = np.nanmean(fr, axis=2, keepdims=True)

    else:
        raise ValueError(f"Unknown baseline method: {method!r}")

    return fr - baseline


def winsorize_trial_axis(
    x: np.ndarray,
    winsor_n: int = 1,
) -> np.ndarray:
    """
    Winsorize values along trial axis.

    Input shape
    -----------
    (n_neuron, n_trial, n_bin)
    """
    if x.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {x.shape}.")

    n_trial = x.shape[1]

    if winsor_n <= 0:
        return x.astype(float, copy=True)

    if 2 * winsor_n >= n_trial:
        raise ValueError(
            f"winsor_n={winsor_n} is too large for n_trial={n_trial}. "
            "Require 2 * winsor_n < n_trial."
        )

    if np.isnan(x).any():
        raise ValueError(
            "Input contains NaN values. Please remove or impute NaNs "
            "before winsorization."
        )

    x_sorted = np.sort(x, axis=1)

    lower = np.take(x_sorted, winsor_n, axis=1)[:, None, :]
    upper = np.take(x_sorted, n_trial - winsor_n - 1, axis=1)[:, None, :]

    return np.clip(x, lower, upper)


def compute_winsorized_t_map(
    x: np.ndarray,
    *,
    winsor_n: int = 1,
    std_floor: float = 1e-8,
) -> np.ndarray:
    """
    Compute winsorized t-like statistic for each neuron and spatial bin.

    Input shape
    -----------
    (n_neuron, n_trial, n_bin)

    Output shape
    ------------
    (n_neuron, n_bin)
    """
    if x.ndim != 3:
        raise ValueError(
            f"Expected x with shape (n_neuron, n_trial, n_bin), "
            f"got {x.shape}."
        )

    n_trial = x.shape[1]

    if n_trial < 2:
        raise ValueError("At least 2 trials are required.")

    x_w = winsorize_trial_axis(
        x,
        winsor_n=winsor_n,
    )

    mean = np.mean(x_w, axis=1)
    std = np.std(x_w, axis=1, ddof=1)
    se = np.maximum(std, std_floor) / np.sqrt(n_trial)

    return mean / se


def get_max_statistic(
    t_map: np.ndarray,
    *,
    alternative: Alternative = "greater",
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract one max statistic across spatial bins for each neuron.
    """
    if t_map.ndim != 2:
        raise ValueError(
            f"Expected t_map with shape (n_neuron, n_bin), "
            f"got {t_map.shape}."
        )

    if alternative == "greater":
        peak_bin = np.argmax(t_map, axis=1)
        t_max = np.take_along_axis(
            t_map,
            peak_bin[:, None],
            axis=1,
        ).ravel()

    elif alternative == "two-sided":
        abs_map = np.abs(t_map)
        peak_bin = np.argmax(abs_map, axis=1)
        t_max = np.take_along_axis(
            abs_map,
            peak_bin[:, None],
            axis=1,
        ).ravel()

    elif alternative == "less":
        peak_bin = np.argmin(t_map, axis=1)
        t_max = -np.take_along_axis(
            t_map,
            peak_bin[:, None],
            axis=1,
        ).ravel()

    else:
        raise ValueError(f"Unknown alternative: {alternative!r}")

    return t_max, peak_bin


def circular_shift_trials(
    x: np.ndarray,
    rng: np.random.Generator,
    *,
    independent_neuron_shift: bool = True,
) -> np.ndarray:
    """
    Circularly shift each trial's spatial firing curve.

    Input shape
    -----------
    (n_neuron, n_trial, n_bin)
    """
    if x.ndim != 3:
        raise ValueError(
            f"Expected x with shape (n_neuron, n_trial, n_bin), "
            f"got {x.shape}."
        )

    n_neuron, n_trial, n_bin = x.shape

    if independent_neuron_shift:
        shifts = rng.integers(
            low=0,
            high=n_bin,
            size=(n_neuron, n_trial),
        )
        indices = (
            np.arange(n_bin)[None, None, :] - shifts[:, :, None]
        ) % n_bin

    else:
        shifts = rng.integers(
            low=0,
            high=n_bin,
            size=(n_trial,),
        )
        indices = (
            np.arange(n_bin)[None, None, :] - shifts[None, :, None]
        ) % n_bin
        indices = np.broadcast_to(indices, x.shape)

    return np.take_along_axis(x, indices, axis=2)


def screen_one_condition(
    fr: np.ndarray,
    *,
    n_merged_bins: int = 40,
    min_trials: int = 10,
    winsor_n: int = 1,
    n_perm: int = 2000,
    alpha: float = 0.05,
    baseline_method: BaselineMethod = "median",
    alternative: Alternative = "greater",
    std_floor: float = 1e-8,
    seed: int | None = 0,
    independent_neuron_shift: bool = True,
) -> dict:
    """
    Screen neurons for one track type and one behavior type.

    If n_trial < min_trials, this condition is skipped.
    """
    if fr.ndim != 3:
        raise ValueError(
            f"Expected fr with shape (n_neuron, n_trial, n_position), "
            f"got {fr.shape}."
        )

    n_neuron, n_trial, _ = fr.shape

    empty_result = {
        "mask": np.zeros(n_neuron, dtype=bool),
        "p_values": np.full(n_neuron, np.nan),
        "t_max": np.full(n_neuron, np.nan),
        "peak_bin": np.full(n_neuron, -1, dtype=int),
        "t_map": None,
        "n_trial": n_trial,
        "n_merged_bins": n_merged_bins,
        "n_perm": n_perm,
        "alpha": alpha,
        "skipped": True,
        "skip_reason": None,
    }

    if n_trial < min_trials:
        empty_result["skip_reason"] = (
            f"n_trial={n_trial} < min_trials={min_trials}"
        )
        return empty_result

    if np.isnan(fr).any():
        empty_result["skip_reason"] = "fr contains NaN"
        return empty_result

    rng = np.random.default_rng(seed)

    fr_merged = merge_position_bins(
        fr,
        n_merged_bins=n_merged_bins,
    )

    x = trial_baseline_correct(
        fr_merged,
        method=baseline_method,
    )

    t_map_real = compute_winsorized_t_map(
        x,
        winsor_n=winsor_n,
        std_floor=std_floor,
    )

    t_max_real, peak_bin = get_max_statistic(
        t_map_real,
        alternative=alternative,
    )

    n_exceed = np.zeros(n_neuron, dtype=int)

    for _ in range(n_perm):
        x_perm = circular_shift_trials(
            x,
            rng=rng,
            independent_neuron_shift=independent_neuron_shift,
        )

        t_map_perm = compute_winsorized_t_map(
            x_perm,
            winsor_n=winsor_n,
            std_floor=std_floor,
        )

        t_max_perm, _ = get_max_statistic(
            t_map_perm,
            alternative=alternative,
        )

        n_exceed += t_max_perm >= t_max_real

    p_values = (n_exceed + 1) / (n_perm + 1)
    mask = p_values < alpha

    return {
        "mask": mask,
        "p_values": p_values,
        "t_max": t_max_real,
        "peak_bin": peak_bin,
        "t_map": t_map_real,
        "n_trial": n_trial,
        "n_merged_bins": n_merged_bins,
        "n_perm": n_perm,
        "alpha": alpha,
        "skipped": False,
        "skip_reason": None,
    }

In [5]:
def correct_across_rules(
    p_values: np.ndarray,
    *,
    alpha: float = 0.01,
    method: RuleCorrection = "bonferroni",
) -> dict:
    """
    Correct p values across selected rules / conditions for each neuron.

    p_values shape
    --------------
    (n_track, n_behavior, n_neuron)
    """
    p_values = np.asarray(p_values, dtype=float)

    if p_values.ndim != 3:
        raise ValueError(
            f"Expected p_values with shape "
            f"(n_track, n_behavior, n_neuron), got {p_values.shape}."
        )

    p_flat = p_values.reshape(-1, p_values.shape[-1])
    valid = np.isfinite(p_flat)

    n_neuron = p_values.shape[-1]
    n_valid_rules = np.sum(valid, axis=0)

    p_min = np.full(n_neuron, np.nan)
    p_neuron = np.full(n_neuron, np.nan)

    for neuron_idx in range(n_neuron):
        ps = p_flat[valid[:, neuron_idx], neuron_idx]

        if len(ps) == 0:
            continue

        ps = np.clip(ps, 1e-300, 1.0)
        p_min[neuron_idx] = np.min(ps)

        if method == "none":
            p_neuron[neuron_idx] = p_min[neuron_idx]

        elif method == "bonferroni":
            p_neuron[neuron_idx] = min(
                p_min[neuron_idx] * len(ps),
                1.0,
            )

        else:
            raise ValueError(
                "method must be 'none' or 'bonferroni'."
            )

    final_mask = (
        np.isfinite(p_neuron)
        & (n_valid_rules > 0)
        & (p_neuron < alpha)
    )

    return {
        "final_mask": final_mask,
        "p_neuron": p_neuron,
        "p_min": p_min,
        "n_valid_rules": n_valid_rules,
        "rule_correction": method,
        "alpha": alpha,
        "n_kept": int(np.sum(final_mask)),
        "keep_rate": float(np.mean(final_mask)),
    }


def screen_loaded_data(
    data: dict,
    *,
    source_key: str = "smooth_firing",
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    n_merged_bins: int = 40,
    min_trials: int = 10,
    winsor_n: int = 1,
    n_perm: int = 2000,
    alpha: float = 0.05,
    baseline_method: BaselineMethod = "median",
    alternative: Alternative = "greater",
    std_floor: float = 1e-8,
    seed: int | None = 0,
    independent_neuron_shift: bool = True,
    rule_correction: RuleCorrection = "bonferroni",
) -> dict:
    """
    Screen neurons across selected track / behavior conditions.

    Responsibility
    --------------
    This function only reads firing-like data and computes neuron-level masks.
    It does not inspect or validate trial index.
    """
    if source_key not in data:
        raise KeyError(f"data does not contain source_key={source_key!r}.")

    track_type = get_data_track_type(data)
    behavior_type = get_data_behavior_type(data)

    firing = data[source_key]

    expected_shape = (
        len(track_type),
        len(behavior_type),
    )

    if firing.shape != expected_shape:
        raise ValueError(
            f"{source_key} shape mismatch. "
            f"Expected {expected_shape}, got {firing.shape}. "
            f"track_type={track_type}, behavior_type={behavior_type}."
        )

    n_neuron = None

    for tt_idx, bt_idx in np.ndindex(firing.shape):
        fr = firing[tt_idx, bt_idx]

        if fr is not None:
            n_neuron = fr.shape[0]
            break

    if n_neuron is None:
        raise ValueError("No valid firing data found.")

    n_track, n_behavior = firing.shape

    condition_mask = np.zeros(
        (n_track, n_behavior, n_neuron),
        dtype=bool,
    )

    p_values = np.full(
        (n_track, n_behavior, n_neuron),
        np.nan,
    )

    t_max = np.full(
        (n_track, n_behavior, n_neuron),
        np.nan,
    )

    peak_bin = np.full(
        (n_track, n_behavior, n_neuron),
        -1,
        dtype=int,
    )

    condition_results = {}
    base_rng = np.random.default_rng(seed)

    for tt_idx, bt_idx in get_itr_index(data, ana_tt, ana_bt):
        tt_name, bt_name = index_to_type(data, tt_idx, bt_idx)

        fr = firing[tt_idx, bt_idx]
        cond_key = f"{tt_name}__{bt_name}"

        if fr is None:
            condition_results[cond_key] = {
                "skipped": True,
                "skip_reason": "fr is None",
                "mask": np.zeros(n_neuron, dtype=bool),
                "p_values": np.full(n_neuron, np.nan),
                "t_max": np.full(n_neuron, np.nan),
                "peak_bin": np.full(n_neuron, -1, dtype=int),
                "n_trial": 0,
            }
            continue

        cond_seed = int(base_rng.integers(0, np.iinfo(np.int32).max))

        res = screen_one_condition(
            fr,
            n_merged_bins=n_merged_bins,
            min_trials=min_trials,
            winsor_n=winsor_n,
            n_perm=n_perm,
            alpha=alpha,
            baseline_method=baseline_method,
            alternative=alternative,
            std_floor=std_floor,
            seed=cond_seed,
            independent_neuron_shift=independent_neuron_shift,
        )

        condition_results[cond_key] = res

        condition_mask[tt_idx, bt_idx, :] = res["mask"]
        p_values[tt_idx, bt_idx, :] = res["p_values"]
        t_max[tt_idx, bt_idx, :] = res["t_max"]
        peak_bin[tt_idx, bt_idx, :] = res["peak_bin"]

    correction_result = correct_across_rules(
        p_values,
        alpha=alpha,
        method=rule_correction,
    )

    final_mask = correction_result["final_mask"]

    return {
        "final_mask": final_mask,
        "condition_mask_uncorrected": condition_mask,
        "p_values": p_values,
        "p_neuron": correction_result["p_neuron"],
        "p_min": correction_result["p_min"],
        "n_valid_rules": correction_result["n_valid_rules"],
        "t_max": t_max,
        "peak_bin": peak_bin,
        "condition_results": condition_results,
        "rule_correction": rule_correction,
        "params": {
            "source_key": source_key,
            "ana_tt": tuple(ana_tt),
            "ana_bt": tuple(ana_bt),
            "n_merged_bins": n_merged_bins,
            "min_trials": min_trials,
            "winsor_n": winsor_n,
            "n_perm": n_perm,
            "alpha": alpha,
            "baseline_method": baseline_method,
            "alternative": alternative,
            "std_floor": std_floor,
            "seed": seed,
            "independent_neuron_shift": independent_neuron_shift,
            "rule_correction": rule_correction,
        },
        "summary": {
            "n_neuron": int(n_neuron),
            "n_kept": int(np.sum(final_mask)),
            "keep_rate": float(np.mean(final_mask)),
        },
    }

In [6]:
def screen_one_file(
    file_path: str | Path,
    *,
    output_root: str | Path,
    gaussian_sigma: int = 0,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    n_merged_bins: int = 40,
    min_trials: int = 10,
    winsor_n: int = 1,
    n_perm: int = 2000,
    alpha: float = 0.05,
    baseline_method: BaselineMethod = "median",
    alternative: Alternative = "greater",
    std_floor: float = 1e-8,
    seed: int | None = 0,
    independent_neuron_shift: bool = True,
    rule_correction: RuleCorrection = "bonferroni",
    overwrite: bool = True,
    skip_existing: bool = False,
) -> tuple[dict, Path]:
    """
    Load one raw file, screen neurons, and save result.

    Boundary
    --------
    - loadData handles raw-data validity.
    - processData handles neuron masking.
    - this function only orchestrates neuron screening.
    """
    file_path = Path(file_path)

    attracted_data = load_raw_data(file_path)

    file_info = add_session_info_to_file_info(
        attracted_data["file_info"],
        file_path,
        require_session_date=True,
    )

    attracted_data["file_info"] = file_info

    save_path = make_screen_save_path(
        output_root,
        file_info,
    )

    if save_path.exists() and skip_existing:
        with open(save_path, "rb") as f:
            result = pickle.load(f)

        return result, save_path

    if save_path.exists() and not overwrite:
        raise FileExistsError(
            f"Screen result already exists: {save_path}. "
            "Use overwrite=True to replace it or skip_existing=True to skip it."
        )

    screening_data = align_track_hpc(
        dict(attracted_data),
        gaussian_sigma=gaussian_sigma,
    )

    screening_result = screen_loaded_data(
        screening_data,
        source_key="smooth_firing",
        ana_tt=ana_tt,
        ana_bt=ana_bt,
        n_merged_bins=n_merged_bins,
        min_trials=min_trials,
        winsor_n=winsor_n,
        n_perm=n_perm,
        alpha=alpha,
        baseline_method=baseline_method,
        alternative=alternative,
        std_floor=std_floor,
        seed=seed,
        independent_neuron_shift=independent_neuron_shift,
        rule_correction=rule_correction,
    )

    final_mask = screening_result["final_mask"]

    masked_attracted_data = apply_neuron_mask_to_attracted_data(
        attracted_data,
        final_mask,
    )

    masked_attracted_data["file_info"] = dict(file_info)

    result = {
        "screening_result": screening_result,
        "final_mask": final_mask,
        "attracted_data": attracted_data,
        "masked_attracted_data": masked_attracted_data,
    }

    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)

    return result, save_path


def _check_unique_mouse_task_session(file_df: pd.DataFrame) -> None:
    """
    Check whether mouse_id + reward_mode + task_type + session_date is unique.
    """
    if "reward_mode" not in file_df.columns:
        file_df = file_df.copy()
        file_df["reward_mode"] = "full"

    required = {
        "mouse_id",
        "reward_mode",
        "task_type",
        "session_date",
    }

    if not required.issubset(file_df.columns):
        missing = required - set(file_df.columns)

        raise ValueError(
            f"file_df is missing required columns: {sorted(missing)}"
        )

    key_cols = [
        "mouse_id",
        "reward_mode",
        "task_type",
        "session_date",
    ]

    duplicated = file_df.duplicated(
        subset=key_cols,
        keep=False,
    )

    if duplicated.any():
        cols = [
            "mouse_id",
            "reward_mode",
            "task_type",
            "session_date",
            "file_name",
            "file_path",
        ]

        dup_df = file_df.loc[
            duplicated,
            [col for col in cols if col in file_df.columns],
        ].sort_values(key_cols)

        raise ValueError(
            "Duplicate mouse_id + reward_mode + task_type + session_date "
            "detected. These files would map to the same screen output.\n"
            f"{dup_df.to_string(index=False)}"
        )


def batch_screen_files(
    file_df: pd.DataFrame,
    *,
    output_root: str | Path,
    gaussian_sigma: int = 0,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    n_merged_bins: int = 40,
    min_trials: int = 10,
    winsor_n: int = 1,
    n_perm: int = 2000,
    alpha: float = 0.05,
    baseline_method: BaselineMethod = "median",
    alternative: Alternative = "greater",
    std_floor: float = 1e-8,
    seed: int | None = 0,
    independent_neuron_shift: bool = True,
    rule_correction: RuleCorrection = "bonferroni",
    check_unique: bool = True,
    skip_existing: bool = False,
    continue_on_error: bool = True,
) -> pd.DataFrame:
    """
    Batch screen all files selected by select_files().
    """
    if "file_path" not in file_df.columns:
        raise ValueError("file_df must contain column 'file_path'.")

    file_df = add_session_columns_to_file_df(
        file_df,
        require_session_date=True,
    )

    if "reward_mode" not in file_df.columns:
        file_df["reward_mode"] = "full"

    if check_unique:
        _check_unique_mouse_task_session(file_df)

    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    rows = []

    metadata_cols = [
        "mouse_id",
        "file_name",
        "task_type",
        "reward_mode",
        "behavior_type_id",
        "session_date",
        "session_key",
        "track_type_id",
        "file_kind",
        "source_stem",
        "suffix",
    ]

    for row_idx, row in file_df.reset_index(drop=True).iterrows():
        file_path = Path(row["file_path"])

        if "file_kind" in row and row["file_kind"] != "mat":
            print(
                f"[{row_idx + 1}/{len(file_df)}] "
                f"Skip non-mat file: {file_path.name}"
            )
            continue

        try:
            result, save_path = screen_one_file(
                file_path,
                output_root=output_root,
                gaussian_sigma=gaussian_sigma,
                ana_tt=ana_tt,
                ana_bt=ana_bt,
                n_merged_bins=n_merged_bins,
                min_trials=min_trials,
                winsor_n=winsor_n,
                n_perm=n_perm,
                alpha=alpha,
                baseline_method=baseline_method,
                alternative=alternative,
                std_floor=std_floor,
                seed=None if seed is None else seed + row_idx,
                independent_neuron_shift=independent_neuron_shift,
                rule_correction=rule_correction,
                overwrite=not skip_existing,
                skip_existing=skip_existing,
            )

            screening_result = result["screening_result"]
            attracted_data = result["attracted_data"]
            file_info = attracted_data["file_info"]

            track_type = get_data_track_type(attracted_data)
            behavior_type = get_data_behavior_type(attracted_data)

            info = {
                "status": "ok",
                "file_path": str(file_path),
                "save_path": str(save_path),
                "n_neuron": screening_result["summary"]["n_neuron"],
                "n_kept": screening_result["summary"]["n_kept"],
                "keep_rate": screening_result["summary"]["keep_rate"],
                "rule_correction": rule_correction,
                "track_type": ",".join(track_type),
                "behavior_type": ",".join(behavior_type),
                "error_type": None,
                "error_message": None,
                "error_traceback": None,
            }

            for col in metadata_cols:
                if col in file_info:
                    info[col] = file_info[col]

            rows.append(info)

            print(
                f"[{row_idx + 1}/{len(file_df)}] OK | "
                f"{file_path.name} | "
                f"mouse={info.get('mouse_id', 'NA')} | "
                f"reward={info.get('reward_mode', 'full')} | "
                f"task_type={info.get('task_type', 'NA')} | "
                f"kept {info['n_kept']}/{info['n_neuron']} "
                f"({info['keep_rate']:.2%}) | "
                f"saved={save_path.name}"
            )

        except Exception as exc:
            error_info = {
                "status": "error",
                "file_path": str(file_path),
                "save_path": None,
                "n_neuron": None,
                "n_kept": None,
                "keep_rate": None,
                "rule_correction": rule_correction,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
                "error_traceback": traceback.format_exc(),
            }

            for col in metadata_cols:
                if col in row:
                    error_info[col] = row[col]

            rows.append(error_info)

            print(
                f"[{row_idx + 1}/{len(file_df)}] ERROR | {file_path}\n"
                f"    {type(exc).__name__}: {exc}"
            )

            if not continue_on_error:
                raise

    summary_df = pd.DataFrame(rows)

    summary_path = output_root / "screen_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    return summary_df

In [7]:
def make_param_grid(param_grid: dict) -> list[dict]:
    """
    Expand parameter grid into a list of configs.
    """
    keys = list(param_grid.keys())
    values = []

    for key in keys:
        value = param_grid[key]

        if isinstance(value, (list, tuple)):
            values.append(list(value))
        else:
            values.append([value])

    configs = []

    for combo in itertools.product(*values):
        configs.append(dict(zip(keys, combo)))

    return configs


def make_config_name(config: dict) -> str:
    """
    Make readable folder name for one parameter setting.
    """
    name_map = {
        "n_merged_bins": "bin",
        "winsor_n": "win",
        "alpha": "alpha",
        "min_trials": "mintr",
        "n_perm": "perm",
        "baseline_method": "base",
        "alternative": "alt",
        "rule_correction": "corr",
    }

    parts = []

    for key, value in config.items():
        short_key = name_map.get(key, key)
        value_str = str(value).replace(".", "p")
        parts.append(f"{short_key}-{value_str}")

    return "__".join(parts)


def run_param_sweep(
    hpc_root,
    *,
    output_root=None,
    task_type=("pattern", "position", "first_pattern", "first_position", "couple"),
    reward_mode=None,
    param_grid=None,
    base_kwargs=None,
):
    """
    Run parameter sweep for selected files.

    Input
    -----
    hpc_root/raw/mouse_id/neuro_type*.mat

    Output
    ------
    hpc_root/screen_param_sweep/config_name/mouse_id/*.pkl
    """
    hpc_root = Path(hpc_root)

    if output_root is None:
        output_root = hpc_root / "screen_param_sweep"

    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    data_root = hpc_root / "raw"

    file_df = select_files(
        data_root=data_root,
        task_type=task_type,
        reward_mode=reward_mode,
        file_kind="mat",
    )

    if len(file_df) == 0:
        raise ValueError(f"No .mat files found under: {data_root}")

    if param_grid is None:
        param_grid = {
            "n_merged_bins": [16, 20, 24, 32],
            "winsor_n": [1],
            "alpha": [0.01],
            "rule_correction": ["none", "bonferroni"],
        }

    configs = make_param_grid(param_grid)

    default_base_kwargs = {
        "gaussian_sigma": 0,
        "ana_tt": ("*",),
        "ana_bt": ("Correct",),
        "min_trials": 10,
        "n_perm": 1000,
        "baseline_method": "median",
        "alternative": "greater",
        "std_floor": 1e-8,
        "seed": 0,
        "independent_neuron_shift": True,
        "check_unique": True,
        "skip_existing": False,
        "continue_on_error": True,
    }

    if base_kwargs is not None:
        default_base_kwargs.update(base_kwargs)

    all_summaries = []

    print(f"Found {len(file_df)} .mat files.")
    print(f"Running {len(configs)} parameter configurations.")

    for config_idx, config in enumerate(configs, start=1):
        run_kwargs = default_base_kwargs.copy()
        run_kwargs.update(config)

        config_name = make_config_name(config)
        config_output_root = output_root / config_name
        config_output_root.mkdir(parents=True, exist_ok=True)

        print("\n" + "=" * 80)
        print(f"[Config {config_idx}/{len(configs)}] {config_name}")
        print(run_kwargs)
        print("=" * 80)

        summary_df = batch_screen_files(
            file_df=file_df,
            output_root=config_output_root,
            **run_kwargs,
        )

        summary_df["config_id"] = config_idx
        summary_df["config_name"] = config_name

        for key, value in run_kwargs.items():
            if isinstance(value, (str, int, float, bool)) or value is None:
                summary_df[key] = value
            else:
                summary_df[key] = str(value)

        all_summaries.append(summary_df)

    all_summary = pd.concat(all_summaries, ignore_index=True)

    all_summary_path = output_root / "all_screen_summary.csv"
    all_summary.to_csv(all_summary_path, index=False)

    return all_summary, file_df


def run_final_screening(
    hpc_root,
    *,
    task_type=("pattern", "position", "first_pattern", "first_position", "couple"),
    reward_mode=None,
    gaussian_sigma: int = 0,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    n_merged_bins: int = 40,
    min_trials: int = 10,
    winsor_n: int = 1,
    n_perm: int = 5000,
    alpha: float = 0.01,
    baseline_method: BaselineMethod = "median",
    alternative: Alternative = "greater",
    std_floor: float = 1e-8,
    seed: int | None = 0,
    independent_neuron_shift: bool = True,
    rule_correction: RuleCorrection = "bonferroni",
    check_unique: bool = True,
    skip_existing: bool = False,
    continue_on_error: bool = True,
):
    """
    Final screening after parameter choice.

    reward_mode
    -----------
    None:
        include both full and 75 files.

    "full":
        only fully rewarded files.

    "75":
        only 75% reward files.
    """
    hpc_root = Path(hpc_root)
    output_root = hpc_root / "screen"
    data_root = hpc_root / "raw"

    file_df = select_files(
        data_root=data_root,
        task_type=task_type,
        reward_mode=reward_mode,
        file_kind="mat",
    )

    if len(file_df) == 0:
        raise ValueError(f"No .mat files found under: {data_root}")

    summary_df = batch_screen_files(
        file_df=file_df,
        output_root=output_root,
        gaussian_sigma=gaussian_sigma,
        ana_tt=ana_tt,
        ana_bt=ana_bt,
        n_merged_bins=n_merged_bins,
        min_trials=min_trials,
        winsor_n=winsor_n,
        n_perm=n_perm,
        alpha=alpha,
        baseline_method=baseline_method,
        alternative=alternative,
        std_floor=std_floor,
        seed=seed,
        independent_neuron_shift=independent_neuron_shift,
        rule_correction=rule_correction,
        check_unique=check_unique,
        skip_existing=skip_existing,
        continue_on_error=continue_on_error,
    )

    summary_path = output_root / "final_screen_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    return summary_df, file_df

In [8]:
def summarize_param_sweep(all_summary: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize keep rate for each parameter configuration.
    """
    config_summary = (
        all_summary
        .groupby("config_name", as_index=False)
        .agg(
            n_files=("file_path", "count"),
            total_neuron=("n_neuron", "sum"),
            total_kept=("n_kept", "sum"),
            mean_keep_rate=("keep_rate", "mean"),
            median_keep_rate=("keep_rate", "median"),
        )
    )

    config_summary["overall_keep_rate"] = (
        config_summary["total_kept"] / config_summary["total_neuron"]
    )

    return config_summary.sort_values("overall_keep_rate")


def make_screen_summary_row(
    *,
    file_path: str | Path,
    save_path: str | Path,
    result: dict,
    rule_correction: str,
) -> dict:
    """
    Make one summary row from screening result.
    """
    screening_result = result["screening_result"]
    attracted_data = result["attracted_data"]
    file_info = attracted_data["file_info"]

    track_type = get_data_track_type(attracted_data)
    behavior_type = get_data_behavior_type(attracted_data)

    row = {
        "file_path": str(file_path),
        "save_path": str(save_path),
        "n_neuron": screening_result["summary"]["n_neuron"],
        "n_kept": screening_result["summary"]["n_kept"],
        "keep_rate": screening_result["summary"]["keep_rate"],
        "rule_correction": rule_correction,
        "track_type": ",".join(track_type),
        "behavior_type": ",".join(behavior_type),
    }

    for col in [
        "mouse_id",
        "file_name",
        "task_type",
        "reward_mode",
        "behavior_type_id",
        "session_date",
        "session_key",
        "track_type_id",
        "file_kind",
        "source_stem",
        "suffix",
    ]:
        if col in file_info:
            row[col] = file_info[col]

    return row


def update_screen_summary(
    summary_path,
    row: dict,
    *,
    key_cols=("mouse_id", "reward_mode", "task_type", "session_date"),
) -> pd.DataFrame:
    """
    Update a screen summary csv by replacing rows with the same key.
    """
    summary_path = Path(summary_path)
    summary_path.parent.mkdir(parents=True, exist_ok=True)

    new_df = pd.DataFrame([row])

    if summary_path.exists():
        old_df = pd.read_csv(summary_path)

        if all(col in old_df.columns for col in key_cols):
            mask = np.ones(len(old_df), dtype=bool)

            for col in key_cols:
                if col in row:
                    mask &= old_df[col].astype(str) == str(row[col])

            old_df = old_df.loc[~mask].copy()

        summary_df = pd.concat(
            [old_df, new_df],
            ignore_index=True,
        )

    else:
        summary_df = new_df

    summary_df.to_csv(summary_path, index=False)

    return summary_df


def infer_hpc_root_from_raw_file(file_path: str | Path) -> Path | None:
    """
    Infer HPC root from a raw file path.

    Example
    -------
    /path/to/HPC_2p/raw/HP01/neuro_type_xxx_pattern.mat
        -> /path/to/HPC_2p
    """
    file_path = Path(file_path).resolve()

    for parent in file_path.parents:
        if parent.name == "raw":
            return parent.parent

    return None


def run_single_file_screening(
    file_path: str | Path,
    *,
    hpc_root: str | Path | None = None,
    output_root: str | Path | None = None,
    gaussian_sigma: int = 0,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    n_merged_bins: int = 40,
    min_trials: int = 10,
    winsor_n: int = 1,
    n_perm: int = 5000,
    alpha: float = 0.01,
    baseline_method: BaselineMethod = "median",
    alternative: Alternative = "greater",
    std_floor: float = 1e-8,
    seed: int | None = 0,
    independent_neuron_shift: bool = True,
    rule_correction: RuleCorrection = "bonferroni",
    overwrite: bool = True,
    update_summary: bool = True,
    summary_name: str = "final_screen_summary.csv",
    skip_existing: bool = False,
) -> tuple[dict, Path, pd.DataFrame | None]:
    """
    Run screening for a single .mat file.
    """
    file_path = Path(file_path)

    if output_root is None:
        if hpc_root is None:
            hpc_root = infer_hpc_root_from_raw_file(file_path)

        if hpc_root is None:
            raise ValueError(
                "Cannot infer hpc_root from file_path. "
                "Please provide hpc_root or output_root explicitly."
            )

        output_root = Path(hpc_root) / "screen"

    output_root = Path(output_root)

    result, save_path = screen_one_file(
        file_path,
        output_root=output_root,
        gaussian_sigma=gaussian_sigma,
        ana_tt=ana_tt,
        ana_bt=ana_bt,
        n_merged_bins=n_merged_bins,
        min_trials=min_trials,
        winsor_n=winsor_n,
        n_perm=n_perm,
        alpha=alpha,
        baseline_method=baseline_method,
        alternative=alternative,
        std_floor=std_floor,
        seed=seed,
        independent_neuron_shift=independent_neuron_shift,
        rule_correction=rule_correction,
        overwrite=overwrite,
        skip_existing=skip_existing,
    )

    row = make_screen_summary_row(
        file_path=file_path,
        save_path=save_path,
        result=result,
        rule_correction=rule_correction,
    )

    summary_df = None

    if update_summary:
        summary_path = output_root / summary_name

        summary_df = update_screen_summary(
            summary_path,
            row,
            key_cols=("mouse_id", "reward_mode", "task_type", "session_date"),
        )

    print(
        f"[Single file screening] "
        f"{row.get('mouse_id', 'NA')} | "
        f"{row.get('session_date', 'NA')} | "
        f"reward={row.get('reward_mode', 'full')} | "
        f"{row.get('task_type', 'NA')} | "
        f"kept {row['n_kept']}/{row['n_neuron']} "
        f"({row['keep_rate']:.2%}) | "
        f"saved to {save_path}"
    )

    return result, save_path, summary_df

In [9]:
summary_df, file_df = run_final_screening(
    hpc_root="../../../data/HPC_2p",
    task_type=("pattern", "position", "first_pattern", "first_position", "couple"),
    reward_mode=None,
    n_merged_bins=32,
    winsor_n=1,
    alpha=0.01,
    rule_correction="bonferroni",
    n_perm=5000,
    min_trials=10,
    baseline_method="median",
    alternative="two-sided",
    ana_tt=("*",),
    ana_bt=("Correct",),
    seed=0,
    skip_existing=True,
    continue_on_error=True,
)

[1/58] OK | neuro_type_saveHP01_1_2024-12-13_first-pattern.mat | mouse=HP01 | reward=full | task_type=first_pattern | kept 1169/1550 (75.42%) | saved=HP01_2024-12-13_first_pattern_screen.pkl
[2/58] OK | neuro_type_saveHP01_1_2024-12-14_pattern.mat | mouse=HP01 | reward=full | task_type=pattern | kept 1154/1292 (89.32%) | saved=HP01_2024-12-14_pattern_screen.pkl
[3/58] OK | neuro_type_saveHP01_1_2024-12-24_position.mat | mouse=HP01 | reward=full | task_type=position | kept 1040/1082 (96.12%) | saved=HP01_2024-12-24_position_screen.pkl
[4/58] OK | neuro_type_saveHP01_3_2024-12-11_couple.mat | mouse=HP01 | reward=full | task_type=couple | kept 1132/1139 (99.39%) | saved=HP01_2024-12-11_couple_screen.pkl
[5/58] OK | neuro_type_saveHP02_1_2024-12-20_couple.mat | mouse=HP02 | reward=full | task_type=couple | kept 692/727 (95.19%) | saved=HP02_2024-12-20_couple_screen.pkl
[6/58] OK | neuro_type_saveHP02_1_2024-12-22_position.mat | mouse=HP02 | reward=full | task_type=position | kept 197/434 (

In [10]:
summary_df[summary_df["status"] == "ok"]

,status,file_path,save_path,n_neuron,n_kept,keep_rate,rule_correction,track_type,behavior_type,error_type,...,file_name,task_type,reward_mode,behavior_type_id,session_date,session_key,track_type_id,file_kind,source_stem,suffix
0,ok,../../../data/HPC_2p/raw/HP01/neuro_type_saveH...,../../../data/HPC_2p/screen/HP01/HP01_2024-12-...,1550,1169,0.754194,bonferroni,"couple_ACB,couple_BCA,CAB,CBA,ACB,BCA,ABC,BAC","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP01_1_2024-12-13_first-pattern...,first_pattern,full,0,2024-12-13,20241213,1,mat,neuro_type_saveHP01_1_2024_12_13_first_pattern,.mat
1,ok,../../../data/HPC_2p/raw/HP01/neuro_type_saveH...,../../../data/HPC_2p/screen/HP01/HP01_2024-12-...,1292,1154,0.893189,bonferroni,"CAB,CBA,ACB,BCA,ABC,BAC","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP01_1_2024-12-14_pattern.mat,pattern,full,0,2024-12-14,20241214,0,mat,neuro_type_saveHP01_1_2024_12_14_pattern,.mat
2,ok,../../../data/HPC_2p/raw/HP01/neuro_type_saveH...,../../../data/HPC_2p/screen/HP01/HP01_2024-12-...,1082,1040,0.961183,bonferroni,"CAB,CBA,ACB,BCA,ABC,BAC","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP01_1_2024-12-24_position.mat,position,full,0,2024-12-24,20241224,0,mat,neuro_type_saveHP01_1_2024_12_24_position,.mat
3,ok,../../../data/HPC_2p/raw/HP01/neuro_type_saveH...,../../../data/HPC_2p/screen/HP01/HP01_2024-12-...,1139,1132,0.993854,bonferroni,"couple_ACB,couple_BCA","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP01_3_2024-12-11_couple.mat,couple,full,0,2024-12-11,20241211,2,mat,neuro_type_saveHP01_3_2024_12_11_couple,.mat
4,ok,../../../data/HPC_2p/raw/HP02/neuro_type_saveH...,../../../data/HPC_2p/screen/HP02/HP02_2024-12-...,727,692,0.951857,bonferroni,"couple_ACB,couple_BCA","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP02_1_2024-12-20_couple.mat,couple,full,0,2024-12-20,20241220,2,mat,neuro_type_saveHP02_1_2024_12_20_couple,.mat
5,ok,../../../data/HPC_2p/raw/HP02/neuro_type_saveH...,../../../data/HPC_2p/screen/HP02/HP02_2024-12-...,434,197,0.453917,bonferroni,"CAB,CBA,ACB,BCA,ABC,BAC","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP02_1_2024-12-22_position.mat,position,full,0,2024-12-22,20241222,0,mat,neuro_type_saveHP02_1_2024_12_22_position,.mat
6,ok,../../../data/HPC_2p/raw/HP02/neuro_type_saveH...,../../../data/HPC_2p/screen/HP02/HP02_2024-12-...,726,565,0.778237,bonferroni,"CAB,CBA,ACB,BCA,ABC,BAC","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP02_1_2024-12-23_position.mat,position,full,0,2024-12-23,20241223,0,mat,neuro_type_saveHP02_1_2024_12_23_position,.mat
7,ok,../../../data/HPC_2p/raw/HP02/neuro_type_saveH...,../../../data/HPC_2p/screen/HP02/HP02_2024-12-...,770,556,0.722078,bonferroni,"couple_ACB,couple_BCA,CAB,CBA,ACB,BCA,ABC,BAC","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP02_2_2024-12-21_first-positio...,first_position,full,0,2024-12-21,20241221,1,mat,neuro_type_saveHP02_2_2024_12_21_first_position,.mat
8,ok,../../../data/HPC_2p/raw/HP02/neuro_type_saveH...,../../../data/HPC_2p/screen/HP02/HP02_2024-12-...,761,478,0.628121,bonferroni,"CAB,CBA,ACB,BCA,ABC,BAC","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP02_2_2024-12-29_pattern.mat,pattern,full,0,2024-12-29,20241229,0,mat,neuro_type_saveHP02_2_2024_12_29_pattern,.mat
9,ok,../../../data/HPC_2p/raw/HP03/neuro_type_saveH...,../../../data/HPC_2p/screen/HP03/HP03_2025-01-...,266,263,0.988722,bonferroni,"couple_ACB,couple_BCA","Correct,FalseAlarm,Miss",None,...,neuro_type_saveHP03_1_2025-01-19_couple.mat,couple,full,0,2025-01-19,20250119,2,mat,neuro_type_saveHP03_1_2025_01_19_couple,.mat


In [11]:
summary_df[summary_df["status"] == "error"][
    ["file_path", "error_type", "error_message"]
]

,file_path,error_type,error_message
